# 데모 2 — Human-in-the-Loop: 위험한 행동 앞에서 멈추기

**한 줄 요약**: 데모 1의 Adaptive RAG 그래프 끝에 "답변 발행"이라는 **비가역적 행동**을
붙이고, 그 직전에 `interrupt()`로 그래프를 멈춰 사람의 **승인/수정/거부**를 받는다.

```mermaid
flowchart LR
    RT{route_question} --> RAG["retrieve → grade → (loop) → generate"]
    RAG --> P{{"publish_answer<br/>⏸ interrupt()"}}
    P -- "approve" --> PUB[("published_answers.md<br/>(외부 발행 가정)")]
    P -- "edit (수정본으로)" --> PUB
    P -- "reject" --> X["발행 취소<br/>+ 사유를 상태에 기록"]
    RT -- "direct (인사말 등)" --> D[direct_answer] --> E((END))
    style P fill:#ffe3e3
```

**before / after**
- before(데모 1): `generate` 가 끝나면 그대로 종료 — 에이전트가 만든 답이 검증 없이 나간다.
- after(지금): 발행 직전에 그래프가 **멈추고 며칠이든 기다릴 수 있다**. 사람의 결정이
  `Command(resume=...)` 로 주입되면 그 지점부터 이어서 실행된다.

필요한 재료는 단 두 가지: **체크포인터**(상태 저장)와 **thread_id**(어느 실행을 재개할지).

In [ ]:
# ── 준비: 데모 1 모듈의 노드들을 그대로 재사용 ────────────────
import os, uuid
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

from demo1_workflow_to_loop.adaptive_rag import (          # 데모 1 s4 구현 재사용!
    RagState, route_question, retrieve, grade_documents, transform_query,
    web_search_node, generate, direct_answer, route_label, decide_after_grade,
)
from common import OUTPUTS
from common.console import banner, step, show_answer, stream_run, console, compare_table
from common.llm import provider
from common.viz import show_graph
from rich.panel import Panel

banner("데모 2 — Human-in-the-Loop", f"LLM_PROVIDER = {provider()}")

PUBLISH_FILE = OUTPUTS / "published_answers.md"

class HitlState(RagState, total=False):
    """RagState + 사람의 결정 기록."""
    decision: str        # approve | edit | reject
    published: bool
    reject_reason: str

## 위험 도구 노드: `publish_answer`

`interrupt(payload)` 는 **노드 안에서** 호출한다 (`interrupt_before` 정적 브레이크포인트가
아니라 이 방식이 공식 권장). 호출 순간 그래프는 예외를 던지며 멈추고, 상태는
체크포인터에 저장된 채 대기한다.

**⚠ 핵심 규칙 (공식 문서)**
1. 재개하면 노드가 **처음부터 다시 실행**된다 → interrupt **이전** 코드는 두 번 실행됨.
2. 그래서 부수 효과(파일 쓰기 = 발행)는 반드시 interrupt **뒤에** 둔다.
3. `interrupt()` 를 bare try/except 로 감싸면 안 된다 (중단 신호 예외를 삼켜버림).
4. payload 는 JSON 직렬화 가능한 값만.

In [ ]:
def publish_answer(state: HitlState) -> dict:
    # ▼ 이 print 는 interrupt '이전' 구간 — 재개 시 한 번 더 찍히는 것을 눈으로 확인!
    console.print("    · publish_answer 진입 [dim](interrupt 이전 구간 — 재개 시 재실행됨)[/]")

    decision = interrupt({                      # ⏸ 여기서 그래프가 멈춘다
        "question": state["question"],
        "draft_answer": state["answer"],
        "actions": ["approve", "edit", "reject"],
    })
    # ── 여기부터는 사람이 응답한 뒤에만 실행된다 ──
    action = decision.get("action", "reject")

    if action == "reject":
        return {"published": False, "decision": "reject",
                "reject_reason": decision.get("reason", "(사유 미기록)")}

    final_answer = state["answer"]
    if action == "edit":
        final_answer = (decision.get("edited_answer") or "").strip() or state["answer"]

    # 부수 효과(발행)는 interrupt 뒤 → 정확히 한 번만 실행된다
    with PUBLISH_FILE.open("a", encoding="utf-8") as f:
        f.write(f"## Q: {state['question']}\n{final_answer}\n(결정: {action})\n\n")
    console.print(f"    · 발행 완료 → {PUBLISH_FILE.name}")
    return {"published": True, "decision": action, "answer": final_answer}

In [ ]:
# ── 그래프 조립: 데모 1 s4 + publish_answer, 그리고 체크포인터 ──
g = StateGraph(HitlState)
for name, fn in [
    ("route_question", route_question), ("retrieve", retrieve),
    ("grade_documents", grade_documents), ("transform_query", transform_query),
    ("web_search", web_search_node), ("generate", generate),
    ("direct_answer", direct_answer),
    ("publish_answer", publish_answer),                    # + 추가된 위험 도구
]:
    g.add_node(name, fn)
g.add_edge(START, "route_question")
g.add_conditional_edges("route_question", route_label, {
    "vectorstore": "retrieve", "web_search": "web_search", "direct": "direct_answer"})
g.add_conditional_edges("grade_documents", decide_after_grade,
                        {"generate": "generate", "transform_query": "transform_query"})
g.add_edge("retrieve", "grade_documents")
g.add_edge("transform_query", "retrieve")
g.add_edge("web_search", "generate")
g.add_edge("generate", "publish_answer")                   # ← 데모1과 달라진 엣지!
g.add_edge("publish_answer", END)
g.add_edge("direct_answer", END)                           # 인사말은 발행 대상 아님 → 그대로 종료

app_hitl = g.compile(checkpointer=MemorySaver())           # ⚠ 체크포인터 필수 (없으면 interrupt 불가)
print("HITL 그래프 컴파일 완료 (checkpointer=MemorySaver)")

In [ ]:
show_graph(app_hitl, "hitl")

In [ ]:
# ── 실행 헬퍼: 중단 감지 → 사람 입력 → Command(resume) 재개 ──
AUTO = os.getenv("DEMO2_AUTO", "").strip().lower()   # ""(대화형) | approve | edit | reject | all
AUTO_EDIT_TEXT = ("interrupt()로 발행 직전에 멈추고, 사람이 Command(resume=...)으로 "
                  "결정을 주입하면 됩니다. — 승인자가 직접 다듬은 답변(수정본)")
AUTO_REJECT_REASON = "근거 문서 인용이 부족해 외부 발행 부적합"

def ask_human(payload: dict, scenario: str) -> dict:
    """interrupt payload 를 사람에게 보여주고 결정을 받는다 (리허설: DEMO2_AUTO 자동응답)."""
    console.print(Panel(
        f"[bold]질문[/]: {payload['question']}\n\n[bold]답변 초안[/]: {payload['draft_answer']}\n\n"
        f"[bold]선택지[/]: {', '.join(payload['actions'])}",
        title="⏸ 사람의 판단 대기 중 (interrupt payload)", border_style="red"))
    mode = scenario if AUTO == "all" else (AUTO or None)
    if mode:
        console.print(f"  [dim]DEMO2_AUTO={AUTO} → '{mode}' 자동 응답 (리허설 모드)[/]")
        if mode == "approve":
            return {"action": "approve"}
        if mode == "edit":
            return {"action": "edit", "edited_answer": AUTO_EDIT_TEXT}
        return {"action": "reject", "reason": AUTO_REJECT_REASON}
    action = (input("action (approve/edit/reject) > ").strip() or "approve").lower()
    decision = {"action": action}
    if action == "edit":
        decision["edited_answer"] = input("수정할 답변 > ")
    if action == "reject":
        decision["reason"] = input("거부 사유 > ")
    return decision

def run_hitl(question: str, scenario: str) -> dict:
    """실행 → interrupt → 사람 결정 → 재개, 전 과정을 한 번에."""
    config = {"configurable": {"thread_id": f"hitl-{uuid.uuid4().hex[:8]}"}}  # 실행마다 독립 스레드
    step(f"시나리오 ({scenario}) — {question[:30]}…")
    final, intr, _ = stream_run(app_hitl, {"question": question, "rewrite_count": 0}, config)
    if not intr:
        show_answer(question, final["answer"], title="interrupt 없이 종료 (발행 대상 아님)")
        return final
    decision = ask_human(intr[0].value, scenario)
    console.print(f"  [bold]▶ Command(resume={decision})[/] 로 재개 ↓")
    final, _, _ = stream_run(app_hitl, Command(resume=decision), config)
    outcome = ("✅ 발행됨" if final.get("published") else f"🚫 발행 취소 — 사유: {final.get('reject_reason')}")
    show_answer(question, final["answer"], title=f"{outcome} (decision={final.get('decision')})")
    return final

## 시나리오 ① 승인 (approve)

데모 1의 "한계 질문"을 그대로 사용한다 — 루프가 돌아 좋은 답을 만들고,
발행 직전에 멈춘 것을 **그대로 승인**한다.
관찰 포인트: 재개 직후 `publish_answer 진입` 로그가 **두 번째로** 찍힌다(노드 재실행).

In [ ]:
r_approve = run_hitl("LangGraph로 human approval 단계를 넣으려면 어떻게 해요?", "approve")

## 시나리오 ② 수정 (edit)

초안이 아쉬울 때 사람이 직접 고친 텍스트로 **교체 후 발행**한다.
resume 값에 `edited_answer` 를 담아 보내면 노드가 그것으로 발행한다.

In [ ]:
r_edit = run_hitl("LangGraph에서 조건부 엣지는 무엇이고 언제 쓰나요?", "edit")

## 시나리오 ③ 거부 (reject)

웹 검색 기반 답변은 근거가 약하다고 판단해 **발행을 취소**하고 사유를 상태에 남긴다.
파일에는 아무것도 추가되지 않는다.

In [ ]:
r_reject = run_hitl("LangGraph 최신 릴리스 소식을 웹에서 검색해 요약해줘", "reject")

In [ ]:
# ── 결과 확인: 발행 파일 + 3경로 상태 비교 ───────────────────
step("결과 확인")
compare_table("HITL 3패턴 결과", ["시나리오", "published", "decision", "비고"], [
    ["approve", str(r_approve.get("published")), r_approve.get("decision", "-"), "초안 그대로 발행"],
    ["edit",    str(r_edit.get("published")),    r_edit.get("decision", "-"),    "수정본으로 발행"],
    ["reject",  str(r_reject.get("published")),  r_reject.get("decision", "-"),
     f"사유: {r_reject.get('reject_reason', '-')}"],
])
assert r_approve.get("published") is True and r_edit.get("published") is True
assert r_reject.get("published") is False and r_reject.get("reject_reason")
print("\n── published_answers.md 마지막 발행 내용 ──")
print("\n".join((PUBLISH_FILE.read_text(encoding="utf-8")).splitlines()[-8:]))

## 정리 — 발표 포인트

1. **interrupt 는 노드 안의 함수 호출 한 줄**이다. UI/알림 시스템이 아니라
   "그래프를 멈추고 상태를 저장하는 원시 연산"이고, 그 위에 Slack 승인 봇이든
   웹 콘솔이든 원하는 UX를 얹으면 된다.
2. **재개 시 노드는 처음부터 재실행된다** — approve 시나리오에서 `publish_answer 진입`
   로그가 두 번 찍힌 것이 증거. 그래서 발행(부수 효과)은 interrupt **뒤에** 두었다.
3. **체크포인터 + thread_id 가 전제 조건** — 상태가 저장되기에 몇 초든 며칠이든
   기다릴 수 있고, thread 별로 독립적인 승인 대기열이 생긴다.

**다음 데모 예고**: 사람이 매번 지켜볼 수는 없다. 에이전트가 실제로 어떤 경로로
무엇을 했는지 **관측(트레이싱)** 하고, 실패를 모아 **개선 루프**를 돌리자 → 데모 3 (Langfuse)